## Load Document

In [1]:
import os
import asyncio
from pathlib import Path
import json
from dotenv import load_dotenv
from langchain_core.documents import Document
from typing import List
import numpy as np
from dotenv import load_dotenv
load_dotenv()

from pinecone import Pinecone, ServerlessSpec

In [2]:
path = Path("../data/contracts")
pdf_paths = list(path.glob("*.pdf"))
processed_pdf = Path("../data/processed_contracts")
processed_pdf

WindowsPath('../data/processed_contracts')

In [ ]:
async def main():
    if not pdf_paths:
        raise ValueError("No PDF files found.")
    
    client = AsyncLlamaCloud(api_key=os.getenv("LLAMA_CLOUD_API_KEY"))

    for pdf_path in pdf_paths:
        print(f"Processing {pdf_path.name}")

        with open(pdf_path, "rb") as f:
            file_obj = await client.files.create(file=f, purpose="parse")

        result = await client.parsing.parse(
            file_id=file_obj.id,
            tier="agentic_plus",
            version="latest",
            output_options={
                "markdown": {
                    "tables": {
                        "output_tables_as_markdown": True
                    }
                }
            },
            expand=["markdown"],
        )

        print(f"Result type: {type(result)}")
        print(f"Markdown: {result.markdown}")
        print(f"Result keys: {result.__dict__.keys()}")

        if result.markdown is None or result.markdown.pages is None:
            print(f"Warning: No markdown returned for {pdf_path.name}, trying text fallback...")
            
            markdown_result = await client.parsing.get_job_markdown(job_id=result.id)
            full_text = markdown_result if isinstance(markdown_result, str) else str(markdown_result)
        else:
            full_text = "\n\n".join(page.markdown for page in result.markdown.pages)

        output_file = processed_pdf / f"{pdf_path.stem}_parsed.txt"
        with open(output_file, "w", encoding="utf-8") as f:
            f.write(full_text)

        print(f"Saved: {output_file}")

await main()

In [3]:
from pydantic import BaseModel, Field
from typing import Optional


class SupplierInfo(BaseModel):
    service_provider_name: Optional[str] = None
    client_name: Optional[str] = None
    agreement_date: Optional[str] = None
    contract_duration: Optional[str] = None
    model_config = {"extra": "forbid"}

class UptimeCommitments(BaseModel):
    guaranteed_uptime_percent: Optional[float] = None
    measurement_period: Optional[str] = None
    excluded_downtime: Optional[str] = None
    model_config = {"extra": "forbid"}

class PenaltyClause(BaseModel):
    clause_id: Optional[str] = None
    trigger_condition: Optional[str] = None
    penalty_type: Optional[str] = None
    penalty_amount: Optional[str] = None
    model_config = {"extra": "forbid"}

class TerminationClauses(BaseModel):
    termination_for_cause: Optional[str] = None
    notice_period_days: Optional[float] = None
    termination_for_convenience: Optional[str] = None
    model_config = {"extra": "forbid"}

class ForceMajeure(BaseModel):
    clause_id: Optional[str] = None
    covered_events: Optional[str] = None
    notification_requirement: Optional[str] = None
    model_config = {"extra": "forbid"}

class DisputeResolution(BaseModel):
    governing_law: Optional[str] = None
    resolution_mechanism: Optional[str] = None
    model_config = {"extra": "forbid"}

class SLADocument(BaseModel):
    supplier_info: Optional[SupplierInfo] = None
    uptime_commitments: Optional[UptimeCommitments] = None
    penalty_clauses: Optional[list[PenaltyClause]] = None
    termination_clauses: Optional[TerminationClauses] = None
    force_majeure: Optional[ForceMajeure] = None
    dispute_resolution: Optional[DisputeResolution] = None
    model_config = {"extra": "forbid"}

In [4]:
schema = SLADocument.model_json_schema()

from llama_cloud import LlamaCloud

client = LlamaCloud(api_key=os.getenv("LLAMA_CLOUD_API_KEY"))

In [9]:
for pdf in pdf_paths:
    print(f"Processing: {pdf.name}")
    with open(pdf, "rb") as f:
        file_obj = client.files.create(file=(pdf.name, f, "application/pdf"), purpose="extract")

    result = client.extraction.extract(
        file_id=file_obj.id,
        data_schema=schema,
        config={},
    )

    # Debug: see what actually came back
    print("Raw result:", result.data)

    sla = SLADocument(**result.data)

    output_path = processed_pdf / f"{pdf.stem}.json"
    output_path.write_text(sla.model_dump_json(indent=2, exclude_none=True), encoding="utf-8")
    print(f"{pdf.name} → {output_path}")

    # Safe access
    if sla.supplier_info:
        print(f"  Provider : {sla.supplier_info.service_provider_name}")
    if sla.uptime_commitments:
        print(f"  Uptime   : {sla.uptime_commitments.guaranteed_uptime_percent}%")
    else:
        print("uptime_commitments not found in document")

Processing: 800263424202323950PMSLA-3208053.pdf
Raw result: {'supplier_info': {'service_provider_name': 'Mazagon Dock Shipbuilders Limited (MDL)', 'client_name': 'Mazagon Dock Shipbuilders Limited', 'agreement_date': None, 'contract_duration': 'two years from the date of placement of Order'}, 'uptime_commitments': None, 'penalty_clauses': None, 'termination_clauses': {'termination_for_cause': 'If the equipment / article / service or any portion thereof be not delivered/ performed by the scheduled delivery date/ period, any stoppage or discontinuation of ordered supply / awarded contract without written consent by Purchaser or not meeting the required quality standards the Purchaser shall be at liberty, without prejudice to the right of the Purchaser to recover Liquidated Damages / penalty as provided for in these conditions or to any other remedy for breach of contract, to terminate the contract either wholly or to the extent of such default.', 'notice_period_days': None, 'termination_

## Chunking

In [4]:
from langchain_community.document_loaders import TextLoader

e:\SentryChain-AI\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [5]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

def split_docs(documents, chunk_size = 2000, chunk_overlap = 500):
    text_splitter = RecursiveCharacterTextSplitter(
        separators=["\nClause", "\nSection", " ", ""],
        chunk_overlap=chunk_overlap,
        chunk_size=chunk_size
    )

    split_docs = text_splitter.split_documents(documents=documents)
    print(f"Splitting {len(documents)} documents into {len(split_docs)} chunks")

    if split_docs:
        print(f"Content {split_docs[0].page_content[:100]}...")

    return split_docs

### Metadata:
* supplier
* section_id
* category(force majeure, penalty, termination)

In [6]:
processed_json_contracts = Path("../data/processed_contracts")
processed_json_contracts_paths = list(processed_json_contracts.glob("*.json"))
processed_json_contracts_paths

[WindowsPath('../data/processed_contracts/800263424202323950PMSLA-3208053.json'),
 WindowsPath('../data/processed_contracts/OnlineSvcsConsolidatedSLA(WW)(English)(January2026)CR (1).json')]

In [7]:
from langchain_community.graphs import Neo4jGraph

In [8]:
from sentence_transformers import SentenceTransformer

In [9]:
class EmbeddingManager:
    def __init__(self, model_name: str = "BAAI/bge-m3" ):
        self.model_name = model_name
        self.model = None
        self.load_model()

    def load_model(self):
        print(f"Loading model: {self.model_name}")
        self.model = SentenceTransformer(self.model_name)
        print(f"Model loaded successfully. Embedding dimension: {self.model.get_sentence_embedding_dimension()}")


    def generate_embeddings(self, texts: List[str]):
        embeddings = self.model.encode(texts, normalize_embeddings=True)
        if len(texts) == 1:
            return embeddings[0].astype(float).tolist()  
        return embeddings.astype(float).tolist()    

embedding_manager = EmbeddingManager()

Loading model: BAAI/bge-m3


Loading weights: 100%|██████████| 391/391 [00:00<00:00, 23884.03it/s]


Model loaded successfully. Embedding dimension: 1024


In [10]:
pc = Pinecone(api_key=os.getenv('PINECONE_API_KEY'))
index_name= "sentry-chain-ai"
if not pc.has_index(index_name):
    pc.create_index(
        name=index_name,
        dimension=1024, 
        metric="cosine",
        spec=ServerlessSpec(
            cloud="aws",
            region="us-east-1"
        )
    )
    print(f"Manual index '{index_name}' created.")

graph = Neo4jGraph(
    url=os.getenv("NEO4J_URI"), 
    username=os.getenv("NEO4J_USERNAME"), 
    password=os.getenv("NEO4J_PASSWORD"),
    database='neo4j'
)

dense_index = pc.Index(index_name)

C:\Users\GOURAV\AppData\Local\Temp\ipykernel_9584\1331176472.py:15: LangChainDeprecationWarning: The class `Neo4jGraph` was deprecated in LangChain 0.3.8 and will be removed in 1.0. An updated version of the class exists in the `langchain-neo4j package and should be used instead. To use it run `pip install -U `langchain-neo4j` and import as `from `langchain_neo4j import Neo4jGraph``.
  graph = Neo4jGraph(


In [ ]:
def data_ingestion(sla_data, contract_id, processed_pdf_path, embedding_manager, dense_index, graph):

    # load supplier name
    supplier_name = sla_data.supplier_info.service_provider_name if sla_data.supplier_info else "Unknown"

    text_file = processed_pdf_path / f"{contract_id}_parsed.txt"
    loader = TextLoader(file_path=str(text_file), encoding="utf-8")
    docs = loader.load()

    contract_chunks = split_docs(documents=docs)

    print("Embedding")
    embedding = [chunk.page_content for chunk in contract_chunks]
    batch_embedding = embedding_manager.generate_embeddings(embedding)

    # supplier -> contract
    graph.query("""
    MERGE (s:Supplier {name: $s_name})
    MERGE (c:Contract {id: $c_id})
    MERGE (s)-[:HAS_CONTRACT]->(c)
    SET c.last_updated = timestamp()
    """, params={
        "s_name": supplier_name,
        "c_id": contract_id
    })

    # Penalty Nodes for Graph Reasoning
    if sla_data.penalty_clauses:
        for idx, p in enumerate(sla_data.penalty_clauses):
            graph.query("""
            MATCH (c:Contract {id: $c_id})
            MERGE (pc:PenaltyRule {id: $p_uid})
            SET pc.type = $type, 
                pc.trigger = $trigger, 
                pc.amount = $amount,
                pc.clause_ref = $ref
            MERGE (c)-[:ENFORCES_PENALTY]->(pc)
            """, params={
                "c_id": contract_id,
                "p_uid": f"rule_{contract_id}_{idx}",
                "type": p.penalty_type,
                "trigger": p.trigger_condition,
                "amount": p.penalty_amount,
                "ref": p.clause_id
            })


    print(f"Upserting {len(contract_chunks)} chunks to Pinecone...")
    for i, (chunk, emb) in enumerate(zip(contract_chunks, batch_embedding)):
        chunk_id = f"{contract_id}#chunk{i}"

        graph.query("""
        MATCH (c:Contract {id: $c_id})
        MERGE (ch:Chunk {vector_id: $v_id})
        SET ch.text_preview = $text
        MERGE (c)-[:HAS_CONTENT_CHUNK]->(ch)
        """, params={
            "c_id": contract_id,
            "v_id": chunk_id,
            "text": chunk.page_content[:200] # Storing preview for graph visualization
        })

        dense_index.upsert(vectors=[{
            "id": chunk_id,
            "values": emb,
            "metadata": {
                **chunk.metadata,
                "text": chunk.page_content,
                "supplier_name": supplier_name,
            }
        }])

In [14]:
for pdf_path in pdf_paths:
    contract_id = pdf_path.stem

    json_path = processed_json_contracts / f"{contract_id}.json"

    with open(json_path, 'r', encoding="utf-8") as f:
        sla_dict = json.load(f)
    
    sla = SLADocument(**sla_dict)

    data_ingestion(sla_data=sla,
                    contract_id=contract_id,
                    processed_pdf_path= processed_pdf,
                    embedding_manager=embedding_manager,
                    dense_index=dense_index,
                    graph=graph)

Splitting 1 documents into 163 chunks
Content # SERVICE LEVEL AGREEMENT (SLA)

Mazagon Dock Shipbuilders Limited invites on-line competitive bids ...
Embedding


[#F5D3]  _: <CONNECTION> error: Failed to read from defunct connection ResolvedIPv4Address(('34.126.64.110', 7687)) (ResolvedIPv4Address(('34.126.64.110', 7687))): OSError('No data')
Unable to retrieve routing information
Transaction failed and will be retried in 0.9285374452794637s (Unable to retrieve routing information)
[#F5D4]  _: <CONNECTION> error: Failed to read from defunct connection IPv4Address(('si-04d2c4a4-8d79.production-orch-0064.neo4j.io', 7687)) (ResolvedIPv4Address(('34.126.64.110', 7687))): OSError('No data')
Transaction failed and will be retried in 1.939486027575205s (Failed to read from defunct connection IPv4Address(('si-04d2c4a4-8d79.production-orch-0064.neo4j.io', 7687)) (ResolvedIPv4Address(('34.126.64.110', 7687))))


Upserting 163 chunks to Pinecone...
Splitting 1 documents into 278 chunks
Content # Service Level Agreement for Microsoft Online Services
# January 1, 2026

# Table of Contents

TABL...
Embedding


[#C4EE]  _: <CONNECTION> error: Failed to read from defunct connection ResolvedIPv4Address(('34.126.64.110', 7687)) (ResolvedIPv4Address(('34.126.64.110', 7687))): OSError('No data')
Unable to retrieve routing information
Transaction failed and will be retried in 0.8239026353703297s (Unable to retrieve routing information)
[#C4EC]  _: <CONNECTION> error: Failed to read from defunct connection IPv4Address(('si-04d2c4a4-8d79.production-orch-0064.neo4j.io', 7687)) (ResolvedIPv4Address(('34.126.64.110', 7687))): OSError('No data')
Transaction failed and will be retried in 1.7694986268746045s (Failed to read from defunct connection IPv4Address(('si-04d2c4a4-8d79.production-orch-0064.neo4j.io', 7687)) (ResolvedIPv4Address(('34.126.64.110', 7687))))


Upserting 278 chunks to Pinecone...


In [ ]:
# for i, chunk in enumerate(chunks):
#     const_id = f"{chunk.metadata['supplier']}#{i}"
#     supplier_name = chunk.metadata.get("supplier_name")

#     # push to neo4j
#     graph.query("""
#     MERGE (s:Supplier {name: $supplier})
#     MERGE (c:Contract {id: $contract_id})
#     SET c.source = $source
#     MERGE (ch:Chunk {vector_id: $v_id})
#     MERGE (s)-[:HAS_CONTRACT]->(c)
#     MERGE (c)-[:HAS_DATA]->(ch)
#     SET ch.text_preview = $text
#     """, params={
#         "supplier": supplier_name,
#         "contract_id": chunk.metadata['supplier_id'],
#         "v_id": const_id,
#         "text": chunk.page_content,
#         "source": chunk.metadata['source']
#     })

#     # push to pinecone
#     vector_values = embedding_manager.generate_embeddings([chunk.page_content])
#     dense_index.upsert(vectors=[{
#         "id": const_id,
#         "values": vector_values,
#         "metadata": {**chunk.metadata, "text": chunk.page_content}
#     }])

In [11]:
from langchain_groq import ChatGroq

groq_llm = ChatGroq(api_key=os.getenv('GROQ_API_KEY'), model="llama-3.3-70b-versatile", temperature=0.1, n=1)

In [12]:
def retriever(query:str, supplier_name: str):
    # Vector search
    query_embedding = embedding_manager.generate_embeddings([query])
    vector_db_result = dense_index.query(
        vector=query_embedding,
        top_k=5,
        
        include_metadata=True,
        filter={"supplier_name": {"$eq": supplier_name}}
    )

    retrieved_vector_ids = [m['id'] for m in vector_db_result['matches']]

    # Graph search
    graph_context = graph.query("""
    MATCH(s:Supplier{name:$supplier_name})-[:HAS_CONTRACT]->(c)-[:HAS_CONTENT_CHUNK]->(ch)
    WHERE ch.vector_id IN $vector_ids  
    RETURN ch.vector_id as id, ch.text_preview as preview
    """, params={
        "supplier_name": supplier_name,  
        "vector_ids": retrieved_vector_ids 
    })

    confirmed_ids = {item['id'] for item in graph_context}

    verified_results = [m for m in vector_db_result['matches'] if m['id'] in confirmed_ids]

    return graph_context, vector_db_result, verified_results

In [13]:
from langchain_core.prompts import PromptTemplate 

In [14]:
## For testing

def rag_query(query:str, contract_id:str):

    json_path = processed_json_contracts / f"{contract_id.replace('_parsed', '')}.json"
    with open(json_path, 'r') as f:
        data = json.load(f)
        supplier_name = data.get("supplier_info", {}).get("service_provider_name", "Unknown")

    graph_context, _, verified_results = retriever(query=query, supplier_name=supplier_name)

    vector_texts = [match['metadata']['text'] for match in verified_results]
    graph_texts = [item['preview'] for item in graph_context]

    combined_contexts = list(set(vector_texts+graph_texts))

    prompt = PromptTemplate(
        template="""You are a contract analyst. Answer the question based ONLY on the provided context.

            Context from {supplier_name}'s SLA:
            {context}

            Question: {question}
            Answer (cite specific clauses when possible):""",
                    input_variables=["supplier_name", "context", "question"]
        )

    formatted_prompt = prompt.format(
        supplier_name=supplier_name,
        context="\n\n".join(combined_contexts),
        question=query
    )

    response = groq_llm.invoke(formatted_prompt)

    return {
        "answer": response.content,
        "sources": [m['id'] for m in verified_results[:3]],
        "context_used": combined_contexts,
        "display_name": supplier_name
    }


result = rag_query(
    query="What are the penalty clauses for service downtime?",
    contract_id="OnlineSvcsConsolidatedSLA(WW)(English)(January2026)CR (1)_parsed"
)

print(result['answer'])
print(f"\nSources: {result['sources']}")
print(f"\n {result['display_name']}")

The penalty clauses for service downtime are outlined in the Service Credit sections throughout the provided context. 

For most services, if the Uptime Percentage is:
- Less than 99.9%, the service credit is 25% (as seen in the sections for Bing Maps Mobile Asset Management, Dynamics 365 Customer Service Enterprise, Dynamics 365 Guides, and Dynamics 365 Intelligent Order Management).
- Less than 99%, the service credit is 50% (as seen in the sections for Bing Maps Mobile Asset Management, Dynamics 365 Customer Service Enterprise, Dynamics 365 Guides, and Dynamics 365 Intelligent Order Management).
- Less than 95%, the service credit is 100% (as seen in the sections for Bing Maps Mobile Asset Management, Dynamics 365 Customer Service Enterprise, Dynamics 365 Customer Insights, and Dynamics 365 Intelligent Order Management).

However, there are some variations:
- For Dynamics 365 Customer Service Enterprise, if the Uptime Percentage is less than 99.99%, the service credit is 5%.
- For D

## Phase 2 News Fetching

In [13]:
target_contract_id = "OnlineSvcsConsolidatedSLA(WW)(English)(January2026)CR (1)_parsed"

In [14]:
json_path = processed_json_contracts / f"{target_contract_id.replace('_parsed', '')}.json"
with open(json_path, 'r') as f:
    data = json.load(f)
    supplier_name = data.get("supplier_info", {}).get("service_provider_name", "Unknown")

In [15]:
from langchain_tavily import TavilySearch, TavilyExtract

def fetch_news(supplier_name, score_threshold:float = 0.5, current_year = "2026"):
    tavily = TavilySearch(api_key=os.getenv('TAVILY_API_KEY'),
                        search_depth="advanced",
                        max_results=5)

    query = f"{supplier_name} supply chain disruption OR outage OR compliance risk OR SLA breach {current_year}"

    results = tavily.invoke(query)

    filtered_results = [r for r in results['results'] if r['score'] >= score_threshold]

    #-----------------------------Input Guardrail-----------------------------------#
    KEYWORDS = ["outage", "breach", "status", "downtime", "incident", "sla", "penalty", "unavailable", "offline", "war"]

    def is_relevant(article):
        text = (article["title"] + article["content"]).lower()
        return any(kw in text for kw in KEYWORDS)

    filtered_results = [a for a in filtered_results if is_relevant(a)]

    return filtered_results

In [16]:
supplier_news_result = fetch_news(supplier_name=supplier_name)

In [17]:
supplier_news_result

[{'url': 'https://medium.com/@ismailkovvuru/microsoft-365-outage-2026-root-cause-forensic-analysis-77712e57b9fc',
  'title': 'Microsoft 365 Outage 2026: Root Cause & Forensic Analysis',
  'content': 'The Harsh Reality:\n\nSLAs are designed to:\n\n1. Limit Microsoft\'s liability (capped at service credits)\n2. Provide marketing comfort ("We guarantee 99.9%!")\n3. Set expectations (some downtime is normal)\n\nSLAs are NOT designed to:\n\n1. Make you whole financially\n2. Prevent outages\n3. Guarantee your business continuity\n\nDesign Around Reality, Not SLA Marketing: [...] [ ] Begin tracking financial impact- [ ] Lost productivity hours- [ ] Lost revenue opportunities- [ ] Overtime costs- [ ] Document for future SLA credit claim---### HOUR 1-4: WORKAROUND STABILIZATIONCommunication Cadence:- [ ] Status updates every 30 minutes to company-wide channel- [ ] Include: What works, what doesn\'t, Microsoft\'s latest update, ETA if knownIT Team:- [ ] Monitor for partial recovery- [ ] Test M36

In [18]:
news_content = "".join([f"Source:{article['title']}\n{article['content']}" for article in supplier_news_result])
news_content

'Source:Microsoft 365 Outage 2026: Root Cause & Forensic Analysis\nThe Harsh Reality:\n\nSLAs are designed to:\n\n1. Limit Microsoft\'s liability (capped at service credits)\n2. Provide marketing comfort ("We guarantee 99.9%!")\n3. Set expectations (some downtime is normal)\n\nSLAs are NOT designed to:\n\n1. Make you whole financially\n2. Prevent outages\n3. Guarantee your business continuity\n\nDesign Around Reality, Not SLA Marketing: [...] [ ] Begin tracking financial impact- [ ] Lost productivity hours- [ ] Lost revenue opportunities- [ ] Overtime costs- [ ] Document for future SLA credit claim---### HOUR 1-4: WORKAROUND STABILIZATIONCommunication Cadence:- [ ] Status updates every 30 minutes to company-wide channel- [ ] Include: What works, what doesn\'t, Microsoft\'s latest update, ETA if knownIT Team:- [ ] Monitor for partial recovery- [ ] Test M365 services every 15 minutes via:  -  (web access)  - Teams app  - OneDrive sync- [ ] Support users transitioning to backup tools- [ ]

In [19]:
import re
import json

In [20]:
def compare_sla(supplier_name : str, news_results : list):
    news_content = "".join([f"Source:{article['title']}\n{article['content']}" for article in news_results])

    query = "service outage uptime penalty SLA breach downtime"
    graph_context, vector_result, verified_results = retriever(query=query , supplier_name=supplier_name)
    vector_texts = [match['metadata']['text'] for match in verified_results]
    graph_texts = [item['preview'] for item in graph_context]

    combined_contexts = list(set(vector_texts+graph_texts))
    prompt = f"""You are a contract risk analyst. 

        NEWS EVENTS:
        {news_content}

        SLA CLAUSES:
        {combined_contexts}

        Based on the news events above, analyze:
        1. Has an SLA violation likely occurred? (Yes/No/Unclear)
        2. Which specific clauses are triggered?
        3. What penalties or remedies apply?
        4. Overall risk severity? (Low/Medium/High/Critical)

        Be specific and cite clause numbers where possible."""

    response = groq_llm.invoke(prompt)

    #------------------------------------Output Guardrail------------------------------------#
    ## Write a prompt for llm to fact check
    VALIDATION_PROMPT = """
    You are a fact checker for legal verdicts.
    
    VERDICT:
    {verdict}
    
    SLA CLAUSES (source of truth):
    {clauses}
    
    Check every specific number, clauses, percentage, amount, or timeframe mentioned in the verdict.
    Does each one actually appear in the SLA clauses above?

    Reply ONLY in this JSON format with no extra text:
    {{"is_grounded": true, "hallucinated_claims": []}}
    """

    validation_input = VALIDATION_PROMPT.format(
        verdict=response.content,
        clauses = "\n".join(combined_contexts)
        )

    ## pass the prompt to llm
    validation_response = groq_llm.invoke(validation_input) 

    try:
        validation_data = json.loads(validation_response.content)
        is_grounded = validation_data.get("is_grounded", False)
        hallucinations = validation_data.get("hallucinated_claims", [])

        if is_grounded:
            print(f"Verdict grounded in SLA clauses")
        else:
            print(f"Hallucinations detected: {hallucinations}")
    except Exception as e:
        print(f"Guardrail could not parse: {e}")
        is_grounded = None
        hallucinations = []

    return {
        "supplier_id": supplier_name,
        "verdict": response.content,
        "is_verified": is_grounded, 
        "hallucinations": hallucinations,
        "news_used": [a['title'] for a in news_results],
        "sla_clauses_matched": combined_contexts
    }

In [21]:
compare_sla(supplier_name=supplier_name, news_results=supplier_news_result)

Hallucinations detected: ['elevated service load during maintenance and a performance issue with a third-party networking provider as the cause of the outage', '25% Service Credit for Uptime Percentage < 99.9% in the verdict does not match the SLA clauses which have different percentages for different services', '50% for Uptime Percentage < 99%, 100% for Uptime Percentage < 95% does not match all the SLA clauses', '90 days’ advance notice of any known significant usage volume increase', 'Scheduled Downtime will not exceed 10 hours per calendar year is specific to Microsoft Cloud App Security', 'High overall risk severity is not mentioned in the SLA clauses', 'The Uptime Percentage calculation clause is not consistently defined across all services', 'The Service Credit clause has varying thresholds across different services']


{'supplier_id': 'Microsoft',
 'verdict': "Based on the news events, here's the analysis:\n\n1. **Yes**, an SLA violation has likely occurred. The Microsoft 365 outage, which lasted for several hours, has affected various services, including Outlook, Microsoft Teams, and Exchange. The outage has resulted in downtime, which is a key metric for SLA calculations.\n\n2. The specific clauses triggered are:\n\t* The Uptime Percentage calculation clause, which states that the Uptime Percentage is calculated using the formula: (User Minutes - Downtime) / User Minutes * 100 (cited in multiple SLA sections, e.g., Exchange Online, Microsoft Cloud App Security).\n\t* The Service Credit clause, which states that if the Uptime Percentage falls below certain thresholds (e.g., 99.9%, 99%, 95%), a Service Credit will be applied (e.g., 25% for Uptime Percentage < 99.9%, 50% for Uptime Percentage < 99%, 100% for Uptime Percentage < 95%) (cited in multiple SLA sections, e.g., Exchange Online, Microsoft Clo

## RAGAS

In [22]:
SYNTHETIC_TEST_CASES = [
    # ── MDL CONTRACT TEST CASES ──────────────────────────────────────────
    {
        "supplier_name": "Mazagon Dock Shipbuilders Limited (MDL)",
        "question": "What is the liquidated damages rate for delay in execution of work under the MDL contract?",
        "ground_truth": (
            "The liquidated damages rate for delay in execution of work is 0.5% (half per cent) "
            "per week or part thereof of the value of the delayed work, subject to a maximum of "
            "5% of the value of delayed work."
        ),
        "relevant_clause": "Clause 20.3 - Liquidated Damages"
    },
    {
        "supplier_name": "Mazagon Dock Shipbuilders Limited (MDL)",
        "question": "What penalty applies if the subcontractor fails to provide the required number of grinders on a given day?",
        "ground_truth": (
            "A penalty of Rs. 200/- per person per shift for grinder and Rs. 250/- per person "
            "per shift for supervisor will be levied and deducted from the subcontractor's invoice "
            "for payment in case of short supply of grinders."
        ),
        "relevant_clause": "Clause 20.2 - Penalty for delay in mobilizing resources"
    },
    {
        "supplier_name": "Mazagon Dock Shipbuilders Limited (MDL)",
        "question": "What is the Security Deposit percentage and within how many days must it be submitted?",
        "ground_truth": (
            "The Security Deposit is 3% of the Order Value (excluding Taxes, Duties, etc.) and "
            "must be submitted within 25 days from the date of intimation of the Order/Contract "
            "in the form of a Bank Guarantee."
        ),
        "relevant_clause": "Clause 17.1 - Security Deposit"
    },
    {
        "supplier_name": "Mazagon Dock Shipbuilders Limited (MDL)",
        "question": "What happens if the Security Deposit Bank Guarantee is not submitted within 25 days?",
        "ground_truth": (
            "If the Security Deposit is not submitted within 25 days, MDL will charge interest "
            "at PLR of SBI + 2% on the SD amount for the delayed period. Additionally, the "
            "contractor may be disqualified/debarred from future MDL tenders and the Order may "
            "be cancelled with invocation of Risk Purchase provisions."
        ),
        "relevant_clause": "Clause 17.6 - Consequences of delay in SD submission"
    },
    {
        "supplier_name": "Mazagon Dock Shipbuilders Limited (MDL)",
        "question": "Within how many days must the subcontractor mobilize workforce after receiving the order?",
        "ground_truth": (
            "The subcontractors are expected to mobilize the workforce and other resources within "
            "14 days from the date of receipt of order. Subcontractor should depute their manpower "
            "within 02 days from the date of job call; delay beyond 02 days shall attract LD."
        ),
        "relevant_clause": "Clause 4.2 - Mobilisation Period"
    },
    {
        "supplier_name": "Mazagon Dock Shipbuilders Limited (MDL)",
        "question": "What is the contract duration for the MDL rate contract?",
        "ground_truth": (
            "The proposed Rate Contract Order shall be valid for a duration of two years from the "
            "date of placement of Order. The tentative contractual period is 13-Apr-2023 to 12-Apr-2025."
        ),
        "relevant_clause": "Clause 4.1 - Contract Duration"
    },
    {
        "supplier_name": "Mazagon Dock Shipbuilders Limited (MDL)",
        "question": "What is the EMD amount required for this MDL tender?",
        "ground_truth": (
            "The Earnest Money Deposit (EMD) applicable for this tender is Rs 10,00,000.00 "
            "(Rs. Ten Lakhs only)."
        ),
        "relevant_clause": "Clause 6.1 - EMD Amount"
    },
    # ── MICROSOFT ONLINE SERVICES SLA TEST CASES ─────────────────────────
    {
        "supplier_name": "Microsoft",
        "question": "What service credit percentage applies when Microsoft Online Services uptime falls below 99.9%?",
        "ground_truth": (
            "When the Monthly Uptime Percentage falls below 99.9%, a service credit of 25% of "
            "the monthly service fee applies. If uptime falls below 99%, the credit increases to "
            "50% of the monthly fee."
        ),
        "relevant_clause": "Uptime SLA and Service Credits table"
    },
    {
        "supplier_name": "Microsoft",
        "question": "What events are excluded from the Microsoft SLA uptime calculation?",
        "ground_truth": (
            "Excluded events include: downtime resulting from customer-initiated operations such "
            "as restart, stop, start, failover, scale compute and storage; factors outside "
            "Microsoft's reasonable control; issues resulting from customer equipment or "
            "third-party services; scheduled maintenance; and downtime from attempts to perform "
            "operations exceeding prescribed quotas or resulting from throttling of suspected "
            "abusive behavior."
        ),
        "relevant_clause": "SLA Exclusions clause"
    },
    {
        "supplier_name": "Microsoft",
        "question": "How is the Monthly Uptime Percentage calculated for Microsoft Online Services?",
        "ground_truth": (
            "The Monthly Uptime Percentage is calculated as: (User Minutes - Downtime) divided by "
            "User Minutes, multiplied by 100. User Minutes is the total accumulated minutes during "
            "a billing month for all Microsoft Online Services. Downtime is the total accumulated "
            "minutes across all service instances where the service was unavailable."
        ),
        "relevant_clause": "Uptime Percentage calculation formula"
    },
    {
        "supplier_name": "Microsoft",
        "question": "What is the process for claiming a service credit from Microsoft after an outage?",
        "ground_truth": (
            "To claim a service credit, the customer must submit a claim within the timeframe "
            "specified in the agreement, typically by filing a support ticket or claim through "
            "the Microsoft admin portal. The claim must include the dates and times of the outage "
            "and the affected services. Microsoft will validate the claim against its own records."
        ),
        "relevant_clause": "Service Credit claim process"
    },
    {
        "supplier_name": "Microsoft",
        "question": "What is the maximum total service credit a customer can receive in a single month?",
        "ground_truth": (
            "The maximum service credit a customer can receive for any billing month is capped "
            "at the total amount paid for the affected service during that billing month. Service "
            "credits are the sole and exclusive remedy for any failure by Microsoft to meet the "
            "service level objectives."
        ),
        "relevant_clause": "Service Credit cap and remedy clause"
    },
]

In [23]:
import time
results = []

for tc in SYNTHETIC_TEST_CASES:
    graph_context, _, verified_results = retriever(
        query=tc["question"],
        supplier_name=tc["supplier_name"]
    )
    
    contexts = [m['metadata']['text'] for m in verified_results]
    
    prompt = f"""Answer this question using only the contract clauses below.

        Question: {tc['question']}

        Clauses:
        {chr(10).join(contexts)}

        Answer:"""

    answer = groq_llm.invoke(prompt).content
    
    results.append({
        "question": tc["question"],
        "expected": tc["ground_truth"],
        "got": answer,
        "contexts": contexts,        
        "contexts_found": len(contexts)
    })
    
    print(f"Q: {tc['question']}")
    print(f"Expected : {tc['ground_truth']}")
    print(f"Got      : {answer[:150]}")
    print(f"Contexts : {len(contexts)}")
    print("-" * 50)

Q: What is the liquidated damages rate for delay in execution of work under the MDL contract?
Expected : The liquidated damages rate for delay in execution of work is 0.5% (half per cent) per week or part thereof of the value of the delayed work, subject to a maximum of 5% of the value of delayed work.
Got      : The liquidated damages rate for delay in execution of work under the MDL contract is 0.5% (half per cent) per week or part thereof, of the delayed wor
Contexts : 5
--------------------------------------------------
Q: What penalty applies if the subcontractor fails to provide the required number of grinders on a given day?
Expected : A penalty of Rs. 200/- per person per shift for grinder and Rs. 250/- per person per shift for supervisor will be levied and deducted from the subcontractor's invoice for payment in case of short supply of grinders.
Got      : According to clause 5(d), if the subcontractor fails to provide the required number of grinders on a given day, a penalty 

In [24]:
from ragas import evaluate
from ragas.metrics import Faithfulness, AnswerRelevancy
from ragas.llms import LangchainLLMWrapper
from langchain_openai import ChatOpenAI
from langchain_community.embeddings import HuggingFaceEmbeddings
from datasets import Dataset
from langchain_google_genai import ChatGoogleGenerativeAI
from ragas.embeddings import LangchainEmbeddingsWrapper 
from ragas.run_config import RunConfig 

# judge_llm = LangchainLLMWrapper(
#     ChatGoogleGenerativeAI(
#         api_key=os.environ["GEMINI_API_KEY"],
#         model="gemini-2.0-flash",
#         temperature=0.1
#     )
# )

judge_llm = LangchainLLMWrapper(groq_llm)

judge_embeddings = LangchainEmbeddingsWrapper(         
    HuggingFaceEmbeddings(model_name="BAAI/bge-m3")
)
dataset = Dataset.from_dict({
    "question": [r["question"] for r in results],
    "answer": [r["got"] for r in results],
    "contexts": [r["contexts"] for r in results],
    "ground_truth": [r["expected"] for r in results],
})


scores = evaluate(
    dataset=dataset,
    metrics=[Faithfulness(), AnswerRelevancy()],
    llm=judge_llm,
    embeddings=judge_embeddings,
    raise_exceptions = False
)

print(scores)
scores.to_pandas()

C:\Users\GOURAV\AppData\Local\Temp\ipykernel_9584\1191855850.py:2: DeprecationWarning: Importing Faithfulness from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import Faithfulness
  from ragas.metrics import Faithfulness, AnswerRelevancy
C:\Users\GOURAV\AppData\Local\Temp\ipykernel_9584\1191855850.py:2: DeprecationWarning: Importing AnswerRelevancy from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import AnswerRelevancy
  from ragas.metrics import Faithfulness, AnswerRelevancy
C:\Users\GOURAV\AppData\Local\Temp\ipykernel_9584\1191855850.py:19: DeprecationWarning: LangchainLLMWrapper is deprecated and will be removed in a future version. Use llm_factory instead: from openai import OpenAI; from ragas.llms import llm_factory; llm = llm_factory('gpt-4o-mini', client=OpenAI(api_key='...'))


{'faithfulness': 0.9479, 'answer_relevancy': 0.9710}


,user_input,retrieved_contexts,response,reference,faithfulness,answer_relevancy
0,What is the liquidated damages rate for delay ...,[17.6. Any delay in submission of SD shall res...,The liquidated damages rate for delay in execu...,The liquidated damages rate for delay in execu...,1.000,NaN
1,What penalty applies if the subcontractor fail...,[20.2. **For delay in mobilizing resources / s...,"According to clause 5(d), if the subcontractor...",A penalty of Rs. 200/- per person per shift fo...,0.750,NaN
2,What is the Security Deposit percentage and wi...,[**Note:** Contractors are requested to raise ...,The Security Deposit percentage is 3% of the O...,The Security Deposit is 3% of the Order Value ...,1.000,0.964631
3,What happens if the Security Deposit Bank Guar...,[17.5. Security Deposit to be submitted in the...,If the Security Deposit Bank Guarantee is not ...,If the Security Deposit is not submitted withi...,1.000,NaN
4,Within how many days must the subcontractor mo...,"[j) Whenever emergency work arises, two days' ...",The subcontractor must mobilize the workforce ...,The subcontractors are expected to mobilize th...,0.750,0.925011
5,What is the contract duration for the MDL rate...,[## 3. INSPECTION: MDL(SQC) & User Department....,The contract duration for the MDL rate contrac...,The proposed Rate Contract Order shall be vali...,1.000,NaN
6,What is the EMD amount required for this MDL t...,[# <u>SERVICE LEVEL AGREEMENT (SLA)</u>\n\n**6...,The EMD amount required for this MDL tender is...,The Earnest Money Deposit (EMD) applicable for...,1.000,0.939302
7,What service credit percentage applies when Mi...,[| Uptime Percentage | Service Credit |\n| ---...,The service credit percentage that applies whe...,When the Monthly Uptime Percentage falls below...,1.000,NaN
8,What events are excluded from the Microsoft SL...,[**Service Level Exceptions:** This SLA does n...,The events excluded from the Microsoft SLA upt...,Excluded events include: downtime resulting fr...,1.000,1.000000
9,How is the Monthly Uptime Percentage calculate...,[$$Monthly\ Uptime\ \% = \frac{(Maximum\ Avail...,The Monthly Uptime Percentage for Microsoft On...,The Monthly Uptime Percentage is calculated as...,0.875,0.996834
